# Preprocess the IEDB peptide-HLA complex data + Feature Extraction 

In [1]:
import pandas as pd
import numpy as np
import os 
import sys
from utils import *
import peptides
peptides.__version__
import peptidy
from utils import build_peptidy_feature_df


/home/hamdaalhosani/Downloads/yes/envs/benchmark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extracting Global Features - via peptides package

In [2]:
# Feature processing for sequences, we will use the peptides package to extract features from the sequences. We will process the sequences in batches to avoid memory issues.
#Parse all the sequences and convert them into feature vectors
def process_batch(seqs_batch):
    return [extract_features(s) for s in seqs_batch]

def feature_processing(seqs, seq_type):
    batch_size = 5000
    all_features = []
    for i in range(0, len(seqs), batch_size):
        print("Finished batch "+str(i))
        batch = seqs[i:i + batch_size]
        batch_features = process_batch(batch)
        all_features.extend(batch_features)
    
    all_features_df = pd.DataFrame(all_features)
    all_features_df.columns = [seq_type+x for x in all_features_df.columns]
    return(all_features_df) 

In [3]:
#Load the unprocessed train data 
unprocessed_train_df = pd.read_csv("../data/IEDB_pHLA_data.csv",header="infer")
unprocessed_train_df.head(5)

,peptide,HLA,immunogenicity,test,respond,potential
0,KEHVFFSEY,HLA-B*4402,Negative,4,0,0.347444
1,DEGATLYRF,HLA-B*4402,Negative,4,0,0.346545
2,TLAARIKFL,HLA-A*0201,Negative,4,0,0.346239
3,KETLNEYKQL,HLA-B*4402,Negative,4,0,0.345162
4,STTDAEACY,HLA-A*0101,Negative,4,0,0.343674


In [4]:
#Convert negative and positive labels to 0 and 1 and keep as a separate column in dataframe
unprocessed_train_df["Label"]=unprocessed_train_df["immunogenicity"].apply(lambda x: 0 if x == "Negative" else 1).astype(int)
unprocessed_train_df["Label"].value_counts()

Label
0    4912
1    4059
Name: count, dtype: int64

In [5]:
#Remove the non-essential columns
dropped_train_df = unprocessed_train_df.drop(columns=["test","respond","potential","immunogenicity"],axis=1)
dropped_train_df.head(5)

#Map the HLA to HLA psuedosequence
dropped_train_df = standardize_hla_alleles(dropped_train_df)
dropped_train_df = map_alleles(dropped_train_df)
dropped_train_df.head(5)
print("Training data shape before: ",dropped_train_df.shape)

#Drop peptides with non-canonical amino acids
#all_peptides_train = dropped_train_df["peptide"].tolist()
#x_peptides_ids = [i for i in range(len(all_peptides_train)) if "X" in all_peptides_train[i]]

#rev_dropped_train_df = dropped_train_df.drop(x_peptides_ids, axis=0 ).reset_index()
#print("Training data shape after: ",rev_dropped_train_df.shape)

all_peptides_train = dropped_train_df["peptide"].tolist()
x_peptides_ids = [i for i in range(len(all_peptides_train)) if ("X" in all_peptides_train[i]) or (not all_peptides_train[i].isupper())]


rev_dropped_train_df = dropped_train_df.drop(x_peptides_ids, axis=0 ).reset_index()
print("Training data shape after: ",rev_dropped_train_df.shape)


#Get the list of all peptides and create feature matrix
all_peptides_train = rev_dropped_train_df["peptide"].tolist()
all_peptides_features_df = feature_processing(all_peptides_train, seq_type="Peptide_")
print(all_peptides_features_df.shape)

all_hla_sequences = rev_dropped_train_df["hla_sequence"].tolist()
all_hla_features_df = feature_processing(all_hla_sequences, seq_type="HLA_")
print(all_hla_features_df.shape)

final_train_df = pd.concat([rev_dropped_train_df, all_peptides_features_df, all_hla_features_df], axis=1)
final_train_df.head(5)

final_train_df["Label"]=final_train_df["Label"].astype(int)
print(final_train_df["Label"].value_counts())
final_train_df.to_csv("../data/processed_data_with_global_features.csv",index=None)

Training data shape before:  (8971, 4)
Training data shape after:  (8968, 5)
Finished batch 0
Finished batch 5000
(8968, 110)
Finished batch 0
Finished batch 5000
(8968, 110)
Label
0    4912
1    4056
Name: count, dtype: int64


In [6]:
processed_train_df = pd.read_csv("../data/processed_data_with_global_features.csv",header="infer")
processed_train_df.head(5)

,index,peptide,HLA,Label,hla_sequence,Peptide_AF1,Peptide_AF2,Peptide_AF3,Peptide_AF4,Peptide_AF5,...,HLA_Z4,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz
0,0,KEHVFFSEY,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.173955,-0.346010,0.376536,-0.138244,-0.186528,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
1,1,DEGATLYRF,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.130313,-0.141878,0.624118,0.427633,0.340208,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
2,2,TLAARIKFL,HLA-A*02:01,0,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,-0.236566,-0.667240,0.421759,0.748899,0.552450,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
3,3,KETLNEYKQL,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.644379,-0.419675,0.461561,0.160180,0.170256,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
4,4,STTDAEACY,HLA-A*01:01,0,YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY,-0.016626,-0.045450,-0.193642,0.402509,-0.348259,...,-0.101471,-0.141471,1.769118,-0.447059,0.085599,4259.78264,63.235294,-7.820588,7.501996,2129.501106


### Preprocess Independent Testing Data (preprocess --> extract features)

In [7]:
#Pre-process the dengue test set and drop unneccessary columns
dengue_test_df = pd.read_csv("../data/independent_tests/dengue_test.csv",header="infer")
dengue_test_df.head(5)

#Remove un-necessary columns
dropped_dengue_test_df = dengue_test_df.drop(columns=["tested","responded","ratio"],axis=1)
dropped_dengue_test_df.head(5)

#Standarize HLA sequences and map them to psuedo sequence
dropped_dengue_test_df = standardize_hla_alleles(dropped_dengue_test_df)
dropped_dengue_test_df = map_alleles(dropped_dengue_test_df)

#Get the list of all peptides and create feature matrix
all_peptides_dengue = dropped_dengue_test_df["peptide"].tolist()
all_peptides_dengue_df = feature_processing(all_peptides_dengue, seq_type="Peptide_")

#Get the list of all HLA and create feature matrix
all_hla_sequences_dengue = dropped_dengue_test_df["hla_sequence"].tolist()
all_hla_sequences_dengue_df = feature_processing(all_hla_sequences_dengue, seq_type="HLA_")

final_dengue_test_df = pd.concat([dropped_dengue_test_df, all_peptides_dengue_df, all_hla_sequences_dengue_df], axis=1)
final_dengue_test_df["immunogenicity"]=final_dengue_test_df["immunogenicity"].astype(int)
print(final_dengue_test_df.head(5))

#Write the processed data frame
#dropped_dengue_test_df.to_csv("../Data/processed_dengue_test.csv",index=None)

final_dengue_test_df.to_csv("../data/independent_tests/processed_dengue_test_with_global_features.csv",index=None)

Finished batch 0
Finished batch 0
      peptide          HLA  immunogenicity  \
0   ETACLGKSY  HLA-A*01:01               1   
1   YAQMWTLMY  HLA-A*01:01               1   
2   YAQMWQLMY  HLA-A*01:01               1   
3  SYAQMWTLMY  HLA-A*01:01               1   
4   TLLCLIPTV  HLA-A*02:01               1   

                         hla_sequence  Peptide_AF1  Peptide_AF2  Peptide_AF3  \
0  YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY    -0.016430     0.040988     0.087939   
1  YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY    -0.234807    -0.391193     0.919612   
2  YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY    -0.127875    -0.447301     0.339797   
3  YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY    -0.234114    -0.212204     0.351687   
4  YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY    -0.761168    -0.065354    -0.109986   

   Peptide_AF4  Peptide_AF5  Peptide_BLOSUM1  ...    HLA_Z4    HLA_Z5  \
0     0.381890     0.193249         0.105556  ... -0.101471 -0.141471   
1    -0.285719     0.407342        -0.647778  ... -0.101471 -0

In [8]:
#Pre-process the sars_cov_2 test set and drop unnecessary columns
sarscov2_test_df = pd.read_csv("../data/independent_tests/sars_cov_2_result.csv",header="infer")
sarscov2_test_df.head(5)

#Remove columns with no value
dropped_sarscov2_test_df = sarscov2_test_df.drop(columns=["id","orf","annotation","convalescent","unexposed",\
                                                          "immunogenicity-con","immunogenicity-un","result","score"],axis=1)
dropped_sarscov2_test_df.head(5)
dropped_sarscov2_test_df["immunogenicity"]=dropped_sarscov2_test_df["immunogenic score"].apply(lambda x: 1 if x > 0.5 else 0)
dropped_sarscov2_test_df = dropped_sarscov2_test_df.drop(columns=["immunogenic score"])

#Standardize the HLA sequences and map them to psuedo sequence
dropped_sarscov2_test_df = standardize_hla_alleles(dropped_sarscov2_test_df)
dropped_sarscov2_test_df = map_alleles(dropped_sarscov2_test_df)

#Get the list of all peptides and create feature matrix
all_peptides_sarscov2 = dropped_sarscov2_test_df["peptide"].tolist()
all_peptides_sarscov2_df = feature_processing(all_peptides_sarscov2, seq_type="Peptide_")

#Get the list of all HLA and create feature matrix
all_hla_sequences_sarscov2 = dropped_sarscov2_test_df["hla_sequence"].tolist()
all_hla_sequences_sarscov2_df = feature_processing(all_hla_sequences_sarscov2, seq_type="HLA_")

final_sarscov2_test_df = pd.concat([dropped_sarscov2_test_df, all_peptides_sarscov2_df, all_hla_sequences_sarscov2_df], axis=1)
final_sarscov2_test_df["immunogenicity"]=final_sarscov2_test_df["immunogenicity"].astype(int)
print(final_sarscov2_test_df.head(5))


#dropped_sarscov2_test_df.to_csv("../Data/processed_sarscov2_test.csv",index=None)

final_sarscov2_test_df.to_csv("../data/independent_tests/processed_sars_cov_2_with_global_features.csv",index=None)

Finished batch 0
Finished batch 0
      peptide          HLA  immunogenicity  \
0  NVFAFPFTIY  HLA-B*15:01               1   
1  IEYPIIGDEL  HLA-B*40:01               0   
2  SELVIGAVIL  HLA-B*40:01               0   
3  GYINVFAFPF  HLA-A*24:02               0   
4   IQYIDIGNY  HLA-B*15:01               1   

                         hla_sequence  Peptide_AF1  Peptide_AF2  Peptide_AF3  \
0  YYAMYREISTNTYESNLYLRYDSYTWAEWAYLWY    -0.482358     0.016638     1.150888   
1  YHTKYREISTNTYESNLYLRYNYYSLAVLAYEWY    -0.090748    -0.066681     0.698623   
2  YHTKYREISTNTYESNLYLRYNYYSLAVLAYEWY    -0.703575    -0.332814    -0.252033   
3  YSAMYEEKVAHTDENIAYLMFHYYTWAVQAYTGY    -0.517564     0.149268     1.062552   
4  YYAMYREISTNTYESNLYLRYDSYTWAEWAYLWY    -0.072878     0.291546     0.950830   

   Peptide_AF4  Peptide_AF5  Peptide_BLOSUM1  ...    HLA_Z4    HLA_Z5  \
0     0.233746     0.301087        -0.593000  ... -0.272059 -0.253235   
1     0.303984    -0.119539        -0.069000  ... -0.376471 -0

In [9]:
#Pre-process the neoantigen test set and drop columns
neoantigen_test_df = pd.read_csv("../data/independent_tests/ori_test_cells.csv",header="infer")
neoantigen_test_df.head(5)

#Remove non-essential columns
dropped_neoantigen_test_df = neoantigen_test_df.drop(columns=["patient","rf_classify","ada_classify","rf_regress","ada_regress",\
                                                     "cnn_classify","cnn_regress","IEDB"],axis=1)
dropped_neoantigen_test_df["immunogenicity"] = dropped_neoantigen_test_df["immunogenic score"].apply(lambda x: 1 if x > 0.5 else 0)
dropped_neoantigen_test_df = dropped_neoantigen_test_df.drop(columns=["immunogenic score"],axis=1)
dropped_neoantigen_test_df.head(5)

#Standardize the HLA sequence and map them to psuedo sequence
dropped_neoantigen_test_df = standardize_hla_alleles(dropped_neoantigen_test_df)
dropped_neoantigen_test_df = map_alleles(dropped_neoantigen_test_df)

all_peptides_neoantigen = dropped_neoantigen_test_df["peptide"].to_list()
all_peptides_neoantigen_df = feature_processing(all_peptides_neoantigen,seq_type="Peptide_")

all_hla_sequences_neoantigen = dropped_neoantigen_test_df["hla_sequence"].to_list()
all_hla_sequences_neoantigen_df = feature_processing(all_hla_sequences_neoantigen, seq_type="HLA_")

final_neoantigen_test_df = pd.concat([dropped_neoantigen_test_df, all_peptides_neoantigen_df, all_hla_sequences_neoantigen_df],axis=1)
final_neoantigen_test_df.to_csv("../data/independent_tests/processed_neoantigen_test_with_global_features.csv",index=None)
final_neoantigen_test_df["immunogenicity"]=final_neoantigen_test_df["immunogenicity"].astype(int)
print(final_neoantigen_test_df.head(5))


#dropped_neoantigen_test_df.to_csv("../Data/processed_neoantigen_test.csv",index=None)
#dropped_neoantigen_test_df.head(5)

Finished batch 0
Finished batch 0
      peptide          HLA  immunogenicity  \
0  ADTSEARPFW  HLA-B*44:02               1   
1  ADVLSPVLVK  HLA-A*03:01               0   
2   AELEEVSSY  HLA-B*44:02               0   
3   AELLAKQLY  HLA-B*44:02               0   
4  AEQQGACPGL  HLA-B*44:02               1   

                         hla_sequence  Peptide_AF1  Peptide_AF2  Peptide_AF3  \
0  YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY     0.108960    -0.058538    -0.375488   
1  YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY    -0.379684    -0.089073    -1.488500   
2  YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY     0.103245    -0.366502    -0.530403   
3  YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY     0.015509    -0.769798    -0.430846   
4  YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY    -0.090521     0.044789    -0.733399   

   Peptide_AF4  Peptide_AF5  Peptide_BLOSUM1  ...    HLA_Z4    HLA_Z5  \
0     0.290965    -0.397001         0.291000  ... -0.595294 -0.155882   
1     0.838277    -1.138928        -0.172000  ... -0.496471 -0

## Extracting local (position-specific) Peptide Features - via peptidy package
#### paper for ref: https://pmc.ncbi.nlm.nih.gov/articles/PMC11961219/

In [10]:
#Load the unprocessed train data 
unprocessed_train_df = pd.read_csv("../data/IEDB_pHLA_data.csv",header="infer")
unprocessed_train_df.head(5)

,peptide,HLA,immunogenicity,test,respond,potential
0,KEHVFFSEY,HLA-B*4402,Negative,4,0,0.347444
1,DEGATLYRF,HLA-B*4402,Negative,4,0,0.346545
2,TLAARIKFL,HLA-A*0201,Negative,4,0,0.346239
3,KETLNEYKQL,HLA-B*4402,Negative,4,0,0.345162
4,STTDAEACY,HLA-A*0101,Negative,4,0,0.343674


In [11]:
#Convert negative and positive labels to 0 and 1 and keep as a separate column in dataframe
unprocessed_train_df["Label"]=unprocessed_train_df["immunogenicity"].apply(lambda x: 0 if x == "Negative" else 1).astype(int)
unprocessed_train_df["Label"].value_counts()

Label
0    4912
1    4059
Name: count, dtype: int64

In [12]:
#Remove the non-essential columns
dropped_train_df = unprocessed_train_df.drop(columns=["test","respond","potential","immunogenicity"],axis=1)
dropped_train_df.head(5)

#Map the HLA to HLA psuedosequence
dropped_train_df = standardize_hla_alleles(dropped_train_df)
dropped_train_df = map_alleles(dropped_train_df)
dropped_train_df.head(5)
print("Training data shape before: ",dropped_train_df.shape)

#Drop peptides with non-canonical amino acids
#all_peptides_train = dropped_train_df["peptide"].tolist()
#x_peptides_ids = [i for i in range(len(all_peptides_train)) if "X" in all_peptides_train[i]]

all_peptides_train = dropped_train_df["peptide"].tolist()
x_peptides_ids = [i for i in range(len(all_peptides_train)) if ("X" in all_peptides_train[i]) or (not all_peptides_train[i].isupper())]


rev_dropped_train_df = dropped_train_df.drop(x_peptides_ids, axis=0 ).reset_index()
print("Training data shape after: ",rev_dropped_train_df.shape)



#  Dataset B: Position-specific peptide + global HLA 



# 1. Extract peptide position-specific features (NEW) ---
peptide_pos_df = build_peptidy_feature_df(
    rev_dropped_train_df,
    peptide_col="peptide",
    padding_len=10
)

print("Peptide position features shape:", peptide_pos_df.shape)


# 2. Extract Global HLA features (just like we did with Dataset A)
all_hla_sequences = rev_dropped_train_df["hla_sequence"].tolist()

all_hla_features_df = feature_processing(
    all_hla_sequences,
    seq_type="HLA_"
)

print("HLA features shape:", all_hla_features_df.shape)


# 3. Concat accirdingly
final_train_df = pd.concat(
    [rev_dropped_train_df, peptide_pos_df, all_hla_features_df],
    axis=1
)

print("Final Dataset B shape:", final_train_df.shape)
final_train_df.head(5)


# 4. fix label accordingly
final_train_df["Label"] = final_train_df["Label"].astype(int)

print(final_train_df["Label"].value_counts())


# 5. dave-
final_train_df.to_csv(
    "../data/processed_data_with_position_specific_features.csv",
    index=False
)

Training data shape before:  (8971, 4)
Training data shape after:  (8968, 5)
Peptide position features shape: (8968, 180)
Finished batch 0
Finished batch 5000
HLA features shape: (8968, 110)
Final Dataset B shape: (8968, 295)
Label
0    4912
1    4056
Name: count, dtype: int64


In [13]:
final_train_df

,index,peptide,HLA,Label,hla_sequence,PeptidePos_p1_f1,PeptidePos_p1_f2,PeptidePos_p1_f3,PeptidePos_p1_f4,PeptidePos_p1_f5,...,HLA_Z4,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz
0,0,KEHVFFSEY,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.0,0.0,0.995649,0.006811,0.0,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
1,1,DEGATLYRF,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.0,0.0,-1.003625,-0.007540,0.0,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
2,2,TLAARIKFL,HLA-A*02:01,0,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,0.0,0.0,-0.004137,-0.000035,0.0,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
3,3,KETLNEYKQL,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.0,0.0,0.995649,0.006811,0.0,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
4,4,STTDAEACY,HLA-A*01:01,0,YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY,0.0,0.0,-0.004137,-0.000039,0.0,...,-0.101471,-0.141471,1.769118,-0.447059,0.085599,4259.78264,63.235294,-7.820588,7.501996,2129.501106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8963,8966,YYMATLKNV,HLA-A*24:02,1,YSAMYEEKVAHTDENIAYLMFHYYTWAVQAYTGY,0.0,1.0,-0.004931,-0.000027,0.0,...,-0.211471,-0.148529,0.893235,-0.361765,-2.820674,4130.57154,54.705882,27.476471,4.574680,2064.926571
8964,8967,ILMNDQEVGV,HLA-A*02:01,1,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,390.0,0.0,-0.004137,-0.000032,1.0,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
8965,8968,ALYEKKLAL,HLA-A*02:01,1,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,100.0,0.0,-0.004137,-0.000046,1.0,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
8966,8969,GIIYIIYKL,HLA-A*02:01,1,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,0.0,0.0,-0.004137,-0.000055,0.0,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
